# EXP 4 — Tail-Anchored Context Bagging (TACB)

exp1/exp3의 비례 stratified 샘플링은 tail을 굶깁니다(cic2018 `web_attacks`: 1M 컨텍스트에 126행).
exp4는 estimator별 컨텍스트를 **명시적으로** 구성합니다:

- **ANCHOR**: pool ≤ `--anchor-threshold`(5,000)인 클래스는 **모든** 컨텍스트에 자연 pool 100% (복제/오버샘플 없음)
- **HEAD**: 나머지 예산을 자연 비례로 채우고 컨텍스트 간 disjoint 분할
- 결합은 확률 평균뿐 — 라우터/게이트 없음 (SRC_HISTORY F1/F6/F9 회피)
- s41(F8)과 구분: head 분포 불변, tail은 자기 자연량 상한 → rebalancing이 아니라 context 구성

같은 run에서 **union 행으로 XGBoost(무가중 + sqrt 가중)** 도 학습 → "floor가 model-agnostic인가"가 즉답됩니다.
`tabpfn_tacb_bp`(balance_probabilities) 컬럼은 덤프된 확률에서 post-hoc으로 자동 계산됩니다.

정직한 프로토콜: `--test-cap-per-class 0` (0818: 100k cap이 web_attacks F1을 0.44 부풀림).

사전 등록 랩 로그: `manuscript/report/0820.md` — run ladder와 성공 기준이 결과 전에 고정되어 있습니다.


## 1. Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 경로
Drive 구조 (이미 만들어 두신 그대로):
```
MyDrive/imbal_cic_tabpfn/
  src/            exp_utils.py, *.ckpt  (+ 아래 .py 2개를 여기 업로드)
  data/           *.pkl
  results/        <- --out-root
  saved_models/   <- --models-dir
```
**src/ 에 올릴 .py 2개**: `nfv3_v3_common.py`, `nfv3_v3_exp3_full_tabpfn_bagging.py`

`exp_utils.py`는 데이터 로더와 클래스별 chronological split 코드라 반드시 필요합니다.


In [ ]:
import os, glob, shutil, sys

DRIVE_ROOT = '/content/drive/MyDrive/imbal_cic_tabpfn'
CODE_DIR   = DRIVE_ROOT + '/src'
DATA_DIR   = DRIVE_ROOT + '/data'
OUT_ROOT   = DRIVE_ROOT + '/results'

# 로컬 작업 트리. 폴더명을 'tabpfn'으로 두면 패키지명과 헷갈리므로 'exp'.
WORK    = '/content/work'
SCRATCH = DRIVE_ROOT + '/tabpfn_cache'   # 세션이 끊겨도 fit 재사용

MODELS_DIR = DRIVE_ROOT + '/saved_models'   # 기본 모드 저장본 ~2-3 GB — Drive OK(업로드 느림)

for d in (SCRATCH, OUT_ROOT, MODELS_DIR, WORK + '/exp', WORK + '/scripts'):
    os.makedirs(d, exist_ok=True)

for label, path in [('CODE_DIR', CODE_DIR), ('DATA_DIR', DATA_DIR)]:
    ok = os.path.isdir(path)
    print(f'{label:10s} {path}   exists={ok}')
    if not ok:
        raise FileNotFoundError(path + ' 가 없습니다. DRIVE_ROOT를 확인하세요.')
print(f'{"MODELS_DIR":10s} {MODELS_DIR}')


## 3. 패키지 설치 — **실패하면 여기서 멈춥니다**
이전 버전은 `!pip -q install ... | tail -1`이라 실패가 조용히 넘어갔고,
그 결과 실행 단계에서 `ModuleNotFoundError: No module named 'tabpfn'`이 났습니다.
여기서는 import까지 확인하고, 버전을 로컬과 맞추려 `tabpfn==8.2.0`으로 고정합니다.


In [ ]:
import importlib, subprocess, sys

def ensure(module, spec):
    try:
        importlib.import_module(module)
        print(f'  {module}: 이미 설치됨')
        return
    except ImportError:
        pass
    print(f'  installing {spec} ...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-4000:]); print(r.stderr[-4000:])
        raise RuntimeError(f'pip install {spec} 실패 — 위 로그를 보세요')
    importlib.invalidate_caches()
    importlib.import_module(module)
    print(f'  {module}: 설치 완료')

ensure('tabpfn', 'tabpfn==8.2.0')
ensure('xgboost', 'xgboost')

import tabpfn, xgboost
print()
print('python  :', sys.executable)
print('tabpfn  :', tabpfn.__version__, '\n          ', tabpfn.__file__)
print('xgboost :', xgboost.__version__)


## 4. GPU 확인 — **T4면 여기서 멈춥니다**


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경에서 GPU를 고르세요.')

name  = torch.cuda.get_device_name(0)
cap   = torch.cuda.get_device_properties(0).total_memory / 1024**3
major = torch.cuda.get_device_capability(0)[0]
print(f'GPU: {name}   {cap:.1f} GiB   sm{major}x')

if major < 8:
    raise RuntimeError(
        f'{name}(sm{major}x)에서는 FlashAttention이 안 돕니다(sm80+ 필요). '
        'MATH 백엔드로 떨어져 메모리가 컨텍스트 길이의 제곱이 되고, 이 노트북의 '
        '메모리 예측이 전부 무효가 됩니다. A100 또는 L4로 런타임을 다시 잡으세요.')
print('OK — 선형 메모리 모델이 성립하는 하드웨어입니다.')

# 조각화 방지: 없으면 들어갈 run도 OOM 납니다 (실측 2.7 GiB 손실)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('PYTORCH_CUDA_ALLOC_CONF =', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


## 5. 코드 배치
v3 스크립트는 `REPO_ROOT/scripts/exp_utils.py`를 import합니다
(`REPO_ROOT` = 스크립트 부모의 부모). 그 구조를 로컬에 만듭니다.
Drive의 `src/` 아래 어느 하위 폴더에 있든 찾아냅니다.


In [ ]:
SCRIPT = 'nfv3_v3_exp4_tail_anchored_bagging.py'
NEEDED = ['nfv3_v3_common.py', SCRIPT, 'exp_utils.py']

missing = []
for name in NEEDED:
    hits = glob.glob(os.path.join(CODE_DIR, '**', name), recursive=True)
    if not hits:
        missing.append(name); continue
    dst = (WORK + '/scripts/' if name == 'exp_utils.py' else WORK + '/exp/') + name
    shutil.copy(hits[0], dst)
    print(f'  {name:32s} <- {hits[0]}')

if missing:
    raise FileNotFoundError(
        f'{CODE_DIR} 아래에서 못 찾은 파일: {missing}\n'
        '로컬 repo의 tabpfn/nfv3_v3_*.py 와 scripts/exp_utils.py 를 업로드하세요.')
print('\nOK')


## 6. 체크포인트
`src/` 안의 `.ckpt`를 찾습니다. 없으면 HF에서 받아 Drive에 캐시합니다.


In [ ]:
CKPT_NAME = 'tabpfn-v3-classifier-v3_20260417_multiclass.ckpt'
hits = glob.glob(os.path.join(CODE_DIR, '**', CKPT_NAME), recursive=True)
if hits:
    CKPT = hits[0]
else:
    from huggingface_hub import hf_hub_download
    src = hf_hub_download(repo_id='Prior-Labs/tabpfn_3', filename=CKPT_NAME)
    CKPT = os.path.join(CODE_DIR, CKPT_NAME)
    shutil.copy(src, CKPT)
print(CKPT, f'{os.path.getsize(CKPT)/1e6:.0f} MB')


## 7. 실행 설정 — run ladder의 어느 run인지 고르세요 (0820.md §3)

In [ ]:
TARGET = 'cic2018'      # cic2018 | ton_iot | bot_iot | *_capped | cic2017_full

# run ladder (0820.md §3, one knob per run):
#   R1a  : E=1, K=1_000_000, ANCHOR=5000            (샘플러 knob만, vs exp1)
#   R3c  : E=4, K=1_000_000, ANCHOR=0,   IPL=True   (E-only 컨트롤)
#   R3a  : E=4, K=1_000_000, ANCHOR=5000, IPL=True  (헤드라인)
#   sweep: K in {100_000, 300_000} x ANCHOR {0, 5000}
E       = 1            # --n-contexts (서로 다른 컨텍스트 수)
NE      = 4            # --n-estimators (총원, E의 배수; exp1과 비교하려면 4 유지)
K       = 1_000_000    # --context-size (estimator당)
ANCHOR  = 5000         # --anchor-threshold (0 = off)
IPL     = (E * K > 1_000_000)   # union이 1M 넘으면 --ignore-pretraining-limits 필요

# 메모리는 컨텍스트 1개 + test 배치 1개 (nfv3_v3_common docstring의 표):
#   cic2018 uncapped test 4.02M rows, K=1M -> TEST_BATCH 220138 (2배치/estimator)
TEST_BATCH = 220138


## 8. 실행
`sys.executable`로 돌립니다 — Colab에서 `python`은 pip가 설치한 인터프리터와
다를 수 있고, 그게 `ModuleNotFoundError: No module named 'tabpfn'`의 원인이었습니다.


In [ ]:
cmd = [
    sys.executable, SCRIPT,
    '--target-dataset', TARGET,
    '--n-contexts', str(E),
    '--n-estimators', str(NE),
    '--context-size', str(K),
    '--anchor-threshold', str(ANCHOR),
    '--test-cap-per-class', '0',        # 정직한 프로토콜 (0818)
    '--test-batch-size', str(TEST_BATCH),
    '--data-dir', DATA_DIR,
    '--out-root', OUT_ROOT,
    '--model-path', CKPT,
    '--resume-dir', SCRATCH,
    '--models-dir', MODELS_DIR,
    '--no-save-models',                 # XGB ckpt는 resume-dir에 남음; TabPFN fit ckpt 없음(0820.md §5)
]
if IPL:
    cmd.append('--ignore-pretraining-limits')
print(' '.join(cmd))


In [ ]:
import subprocess, sys

def run(command):
    """Stream the run, capture its output dir, FAIL LOUDLY on a bad exit.

    An earlier version of this notebook globbed for the newest results dir
    afterwards, so a crashed run silently displayed a PREVIOUS run's numbers.
    """
    p = subprocess.Popen(command, cwd=WORK + '/exp', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ})
    captured = []
    for line in p.stdout:
        sys.stdout.write(line)
        captured.append(line)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(
            f'실행 실패 (exit code {p.returncode}) — 위 로그를 보세요. '
            '아래 결과 셀은 실행하지 마세요(예전 run을 읽게 됩니다).')
    out = [l.split('Wrote ', 1)[1].strip() for l in captured if l.startswith('Wrote ')]
    if not out:
        raise RuntimeError("'Wrote <dir>' 줄이 없습니다 — 아티팩트가 안 만들어졌습니다.")
    return out[-1]

RUN_DIR = run(cmd)
print('\nrun dir:', RUN_DIR)


## 9. 결과
위 셀이 만든 **바로 그** 디렉터리(`RUN_DIR`)만 읽습니다.


In [ ]:
import pandas as pd, json

t = pd.read_csv(os.path.join(RUN_DIR, 'per_class_metrics.csv'))
avg = ['macro_avg', 'weighted_avg', 'tail_avg']
display(t[t['class'].isin(avg)].pivot(index='class', columns='method', values='f1'))
print()
display(t[~t['class'].isin(avg)].pivot(index='class', columns='method',
                                       values=['f1', 'support']))
print()
comp = pd.read_csv(os.path.join(RUN_DIR, 'context_composition.csv'))
display(comp[comp['estimator'].astype(str) == '0'])   # estimator 0의 구성 (anchor/head/rows)


In [ ]:
print(json.dumps(json.load(open(os.path.join(RUN_DIR, 'timings.json')))[0], indent=2))


`per_class_metrics.csv` · `split_audit.csv` · `train_test_drift_diagnostic.csv` ·
`args.json` · `timings.json` 가 `RUN_DIR`에 저장됩니다.

⚠️ `train_test_drift_diagnostic.csv`의 `median_abs_z`는 `small_train_warning=True`인
행에서 신뢰할 수 없습니다(표준편차를 수십 행에서 추정). `manuscript/report/0817.md` 참조.


## 10. 기록된 exp1 컨트롤과 비교 (재실행 없음)
exp1(`20260818_*_exp1_1m`, 비례 1M 컨텍스트)이 R1a의 one-knob 짝입니다.
`args.json` 비교로 knob이 하나만 다름을 확인하세요.


In [ ]:
EXP1_DIR = {   # uncapped-test exp1 runs (기록)
    'cic2018':  OUT_ROOT + '/20260818_042459_nfv3_cic2018_exp1_1m',
    'bot_iot':  OUT_ROOT + '/20260818_053139_nfv3_botiot_exp1_1m',
    'ton_iot':  OUT_ROOT + '/20260818_071152_nfv3_toniot_exp1_1m',
    'cic2017_full': OUT_ROOT + '/20260818_030904_cic2017_full_exp1_1m',
}[TARGET]
a = pd.read_csv(os.path.join(EXP1_DIR, 'per_class_metrics.csv'))
b = pd.read_csv(os.path.join(RUN_DIR, 'per_class_metrics.csv'))
cmpdf = (a[a['method'] == 'tabpfn_v3'].set_index('class')['f1'].rename('exp1_tabpfn_prop').to_frame()
    .join(a[a['method'] == 'xgboost'].set_index('class')['f1'].rename('exp1_xgb_prop'))
    .join(b[b['method'] == 'tabpfn_tacb'].set_index('class')['f1'].rename('exp4_tacb'))
    .join(b[b['method'] == 'tabpfn_tacb_bp'].set_index('class')['f1'].rename('exp4_tacb_bp'))
    .join(b[b['method'] == 'xgboost'].set_index('class')['f1'].rename('exp4_xgb_union'))
    .join(b[b['method'] == 'xgboost_sqrt'].set_index('class')['f1'].rename('exp4_xgb_sqrt')))
cmpdf['tacb_minus_exp1xgb'] = cmpdf['exp4_tacb'] - cmpdf['exp1_xgb_prop']
display(cmpdf)


판정 규칙 (0820.md §3, 사전 등록):
- exp4_xgb_*가 이미 bar에 도달 → 효과는 **model-agnostic** (floor가 기여) → 서사 피벗
- exp4_tacb가 XGB보다 **+0.05 이상 더** 얻음 → R3 (E=4 bagging) 진행
- 아무것도 안 움직임 → shift/overlap 한계 → 소예산 스윕 + `--eval-split val` 축이 본론

성공 기준: web_attacks F1 ≥ bar+0.05, macro ≥ exp1 XGB−0.010, benign ≥ −0.005, 비-tail 손실 ≤0.02.
